In [3]:
!pip install -q transformers datasets peft evaluate scikit-learn

In [4]:
!pip install -q --upgrade torchao

In [5]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType
import evaluate
import numpy as np

# Verificamos si la GPU está encendida
print(f"¿GPU disponible?: {torch.cuda.is_available()}")

¿GPU disponible?: True


In [12]:
# 1. Cargar los CSV
dataset = load_dataset('csv', data_files={'train': 'ag_news_train.csv', 'test': 'ag_news_test.csv'})

# 2. Cargar el Tokenizador
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# 3. Función de tokenización (Padding y Truncation)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Aplicar tokenización a todo el dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# --- EL FIX MAESTRO DE LAS ETIQUETAS ---
# Detectamos los textos únicos ('Sports', 'Business'...) y les asignamos (0, 1, 2, 3)
etiquetas_unicas = dataset['train'].unique('label')
diccionario_etiquetas = {nombre: i for i, nombre in enumerate(etiquetas_unicas)}

def cambiar_a_numeros(ejemplo):
    ejemplo['label'] = diccionario_etiquetas[ejemplo['label']]
    return ejemplo

# Aplicamos la traducción
tokenized_datasets = tokenized_datasets.map(cambiar_a_numeros)
# ---------------------------------------

# Ajustar las columnas para el modelo
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [13]:
# Cargar el modelo base congelado
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=4)

# Configurar LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, # Rango
    lora_alpha=16, # Factor de escala
    target_modules=["q_lin", "v_lin"], # Dónde inyectar LoRA en DistilBERT
    lora_dropout=0.1
)

# Envolver el modelo base con LoRA
peft_model = get_peft_model(model, lora_config)

# Imprimir cuántos parámetros vamos a entrenar (Debería ser menos del 1%)
peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948


In [14]:
# Métrica para evaluar
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Usamos macro porque es un problema multiclase balanceado
    return metric.compute(predictions=predictions, references=labels, average="macro")

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./lora_ag_news",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    label_names=["labels"] # <---Obliga al Trainer a usar las etiquetas
)

# Apagamos la búsqueda de dependencias de video que causa error
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

# Inicializar el Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# ¡ARRANCAR EL ENTRENAMIENTO!
print("Iniciando Fine-Tuning con LoRA...")
trainer.train()

Iniciando Fine-Tuning con LoRA...


Epoch,Training Loss,Validation Loss,F1
1,0.286319,0.290045,0.906642
2,0.271204,0.280574,0.908182
3,0.246941,0.265615,0.912217


TrainOutput(global_step=1500, training_loss=0.2921109898885091, metrics={'train_runtime': 228.5226, 'train_samples_per_second': 105.022, 'train_steps_per_second': 6.564, 'total_flos': 808493137920000.0, 'train_loss': 0.2921109898885091, 'epoch': 3.0})